
# <span style="color:rgb(213,80,0)">Examen P1 de robótica</span>

# <span style="color:rgb(213,80,0)">Modelado cinemático y dinámico de un robot RRR</span>

## Introduccion

La robótica moderna exige un nivel de precisión y eficiencia que solo puede alcanzarse a través de un entendimiento profundo de la física y la matemática que rigen el movimiento. En este reporte, se analiza un **manipulador robótico plano de tres grados de libertad (3-DOF)** con una configuración cinemática **RRR** (Rotacional-Rotacional-Rotacional). 

Este sistema, compuesto por tres eslabones de longitudes $L_1, L_2, L_3$ y tres articulaciones angulares $q_1, q_2, q_3$, representa uno de los modelos fundamentales para comprender la interacción entre el espacio articular y el espacio cartesiano en un entorno bidimensional.

A diferencia de los manipuladores de dos grados de libertad, la configuración 3-DOF en un plano introduce una **redundancia cinemática** si solo se considera la posición final $(x, y)$, o bien, permite el control total de la **orientación del efector final** ($\phi$). Esto lo convierte en uno de los modelos fundamentales para comprender la compleja interacción entre el **espacio articular** (ángulos de los motores) y el **espacio cartesiano** (posición y orientación en el mundo real) en un entorno bidimensional.

El análisis aquí presentado no solo aborda la geometría del movimiento, sino que sienta las bases para el estudio de la maniobrabilidad y la destreza del robot dentro de su espacio de trabajo, elementos críticos para aplicaciones que van desde el ensamblaje industrial hasta la asistencia médica de precisión.

#### Justificación Técnica

En la ingeniería actual, el diseño de un robot no concluye con su construcción física; la creación de un "Gemelo Digital" o modelo matemático es indispensable. 

* El **modelado cinemático** (postura y velocidad) permite traducir las rotaciones de los motores en coordenadas precisas del efector final, lo cual es vital para tareas de alta fidelidad como el microensamblaje o la cirugía asistida.
* Por otro lado, el **modelado dinámico** es esencial para el control de fuerzas y la optimización energética, permitiendo predecir los torques necesarios para mover el brazo bajo diferentes condiciones de carga. 

Sin estos modelos, la simulación y el control seguro de robots en entornos industriales serían imposibles.


#### 1. Cinemática: ¿Dónde estoy? vs. ¿Cómo llego?
*   **Modelo Directo (Forward Kinematics):**
    *   **Concepto:** Calcula la posición del extremo del robot a partir de los ángulos de los motores.
    *   **Física:** Representa la lectura de sensores (**encoders**) para localizar al robot en el espacio. Es un cálculo sencillo con una única solución.
*   **Modelo Inverso (Inverse Kinematics):**
    *   **Concepto:** Calcula los ángulos necesarios para alcanzar una posición deseada.
    *   **Física:** Es el **cerebro planeando el movimiento**. Es complejo porque un robot (ej. RRR) puede tener múltiples configuraciones (codo arriba/abajo) para un mismo punto, o ninguna si está fuera de alcance.

---

#### 2. Dinámica: Movimiento vs. Fuerza
*   **Dinámica Directa:**
    *   **Concepto:** Determina el movimiento resultante al aplicar torques específicos.
    *   **Física:** Describe la **reacción natural** del brazo; por ejemplo, cómo cae por gravedad e inercia si se liberan los frenos.
*   **Dinámica Inversa:**
    *   **Concepto:** Determina el torque necesario para lograr un movimiento específico.
    *   **Física:** Es la base del **control de potencia**. Indica cuánta corriente enviar a los motores para vencer peso y fricción, manteniendo la velocidad y aceleración deseadas.


#### Objetivo del Reporte

El objetivo primordial de este trabajo es obtener y validar el modelo matemático integral del manipulador 3-DOF. De forma específica, se pretende:

*   Derivar las **ecuaciones de postura** (cinemática directa) para localizar el punto final $\{ P \}$ en el plano $XY$.
*   Calcular la **matriz Jacobiana** para determinar los perfiles de velocidad y aceleración.
*   Desarrollar el **modelo dinámico** que relacione los movimientos con las fuerzas y torques aplicados en cada articulación.

---

#### Hipótesis

Se plantea que, mediante la aplicación sistemática de la **convención de Denavit-Hartenberg** y el **formalismo de Euler-Lagrange**, es posible describir con absoluta fidelidad el comportamiento del robot en un entorno simulado. Se espera que los modelos resultantes garanticen una transición fluida entre trayectorias, permitiendo que el efector final alcance cualquier punto dentro de su espacio de trabajo con un error nulo en condiciones ideales de simulación.

---

#### Vinculación con la Clase

Este reporte es la culminación práctica de los temas abordados durante el primer parcial. Se integran conceptos clave como las **Matrices de Transformación Homogénea** para la localización de marcos de referencia, el análisis de **velocidades mediante Jacobianos** y las leyes de la **mecánica clásica** aplicadas a sistemas multicuerpo. Este ejercicio permite aterrizar la teoría abstracta de la clase en una aplicación técnica real, preparándonos para los retos del control de robots en el siguiente nivel de la asignatura.


### Definición de funciones

In [1]:
%Deficición de la función de manera simbolica
syms Tij(x_i_j,y_i_j,z_i_j,gi_j,bi_j,ai_j)

%Definición de la transformación homógenea general
Tij(x_i_j,y_i_j,z_i_j,gi_j,bi_j,ai_j) = [cos(ai_j)*cos(bi_j) cos(ai_j)*sin(bi_j)*sin(gi_j)-sin(ai_j)*cos(gi_j) sin(ai_j)*sin(gi_j)+cos(ai_j)*sin(bi_j)*cos(gi_j) x_i_j; sin(ai_j)*cos(bi_j) cos(ai_j)*cos(gi_j)+sin(ai_j)*sin(bi_j)*sin(gi_j) sin(ai_j)*sin(bi_j)*cos(gi_j)-cos(ai_j)*sin(gi_j) y_i_j; -sin(bi_j) cos(bi_j)*sin(gi_j) cos(bi_j)*cos(gi_j) z_i_j; 0 0 0 1]

## Modelado del robot RRR


![Robot_RRR](imagenes_ep1/Robot_RRR.png)

Fig. 1. Robot RRR, se muestra como ejemplo para obtener su modelado matematico.

## Planteamiento del modelo cinemático de la posición

El análisis cinemático de posición de un robot **RRR** es el proceso matemático que permite establecer la relación funcional entre las variables articulares (ángulos $q_1, q_2, q_3$) y la pose final del efector ($x, y, \phi$) en el espacio de trabajo.

![postura1](imagenes_ep1/postura1.png)

Fig. 2. Cadena cinematica y sus componentes, para obtener el modelo cinemático de la posición.


In [2]:
syms x_O_1 y_O_1 theta_O_1 L_2 theta_1_2 L_3 theta_2_3 L_1

T_O_1 = Tij(x_O_1,y_O_1,0,0,0,theta_O_1)

In [3]:
T_1_2 = Tij(L_1,0,0,0,0,theta_1_2)

In [4]:
T_2_3 = Tij(L_2,0,0,0,0,theta_2_3)

In [5]:
T_3_P = Tij(L_3,0,0,0,0,0)

In [6]:

T_O_P = simplify(T_O_1*T_1_2*T_2_3*T_3_P)

#### Planteamiento del modelo de la postura

El **modelo de la postura** (o cinemática de posición) es la descripción matemática que define la ubicación y orientación completa del efector final del robot en un sistema de coordenadas de referencia. Su importancia radica en los siguientes puntos:

*   **¿Para qué sirve?**: Es la herramienta que permite al sistema de control transformar las coordenadas deseadas en el espacio cartesiano (ej. "mover a la coordenada $x=10, y=5$") en comandos específicos para cada motor del robot. Sin este modelo, el robot no tendría consciencia de su propia ubicación en el mundo físico.
*   **¿Qué representa?**: Físicamente, representa la **pose** del robot. En un manipulador plano, esto incluye:
    *   **Posición ($x, y$):** El punto exacto en el plano donde se encuentra la punta o herramienta.
    *   **Orientación ($\phi$):** El ángulo que forma el último eslabón respecto a un eje fijo, lo cual es crucial para tareas de agarre o alineación de precisión.

En términos matemáticos, se suele expresar como una **Matriz de Transformación Homogénea ($T$)** que consolida la rotación y la traslación del robot en una sola entidad.


$$^O \xi_P =^O \xi_P (q)$$

$$q=\lbrace^0 \theta_1 ,^1 \theta_2 ,^2 \theta_3 \rbrace$$


In [7]:
syms P_0_P
P_0_P = [T_O_P(1,4);T_O_P(2,4);0]

![postura2](imagenes_ep1/postura2.png)

Fig. 3. Vector de posicion P_0_p

In [8]:
syms theta_O_P
theta_O_P = [theta_O_1+theta_1_2+theta_2_3]

#### Vector de postura del robot

El **Vector de Postura** es la representación matemática mínima y suficiente de la ubicación y orientación del efector final del robot en el espacio de trabajo.

*   Físicamente, consolida en una sola entidad matemática (generalmente un vector columna $\mathbf{p}$) los datos espaciales críticos:
    *   **Posición ($x, y$):** Las coordenadas cartesianas del punto de interés (la punta del robot).
    *   **Orientación ($\phi$):** El ángulo de inclinación de la herramienta respecto a un eje de referencia.
*   **¿Para qué sirve?**: 
    1.  **Estado del Sistema:** Actúa como la "salida" del modelo cinemático directo, permitiendo conocer la ubicación exacta del robot en cada instante de tiempo $t$.
    2.  **Referencia de Control:** Es el valor que el controlador compara con la "pose deseada" para calcular el error y corregir el movimiento de los motores.
    3.  **Generación de Trayectorias:** Facilita la programación de movimientos complejos, como seguir una línea recta o un arco, al definir los puntos de paso en términos de coordenadas mundiales.

In [9]:
syms xi_O_P_q
xi_O_P_q = [T_O_P(1,4);T_O_P(2,4);theta_O_1+theta_1_2+theta_2_3]


Sustituyendo


$$^0 \xi_P (q)=\left(\begin{array}{c} P_{O,P} \newline \theta_{O,P}  \end{array}\right)=\left(\begin{array}{c} x_{O,1} +x_{2,3} \,\cos \left(\theta_{1,2} +\theta_{O,1} \right)+x_{1,2} \,\cos \left(\theta_{O,1} \right)+x_{3,P} \,\cos \left(\theta_{1,2} +\theta_{2,3} +\theta_{O,1} \right)\newline y_{O,1} +x_{3,P} \,\sin \left(\theta_{1,2} +\theta_{2,3} +\theta_{O,1} \right)+x_{2,3} \,\sin \left(\theta_{1,2} +\theta_{O,1} \right)+x_{1,2} \,\sin \left(\theta_{O,1} \right)\newline \theta_{1,2} +\theta_{2,3} +\theta_{O,1}  \end{array}\right)$$

In [10]:
syms x_O_P y_O_P theta_O_P
xi_O_P = [x_O_P; y_O_P; theta_O_P]


Sustituyendo


$$F=^0 \xi_P -^0 \xi_P (q)=0$$

$$^O \xi_P =^O \xi_P (q)$$
## Modelado cinemático de las velocidades

El **Modelado Cinemático de las Velocidades** es el análisis que establece la relación entre la rapidez de movimiento de las articulaciones y la velocidad lineal y angular del efector final. Su importancia se resume en los siguientes puntos:

*   **Representación Matemática**: Se representa mediante la **Matriz Jacobiana** ($J$), la cual actúa como un operador de transformación lineal que mapea las velocidades articulares ($\dot{q}$) al espacio de velocidades cartesianas ($\dot{x}$).
*   **Utilidad Técnica**: 
    1.  **Control en Tiempo Real**: Permite determinar las velocidades que deben alcanzar los motores para que el robot siga una trayectoria con una rapidez específica.
    2.  **Identificación de Singularidades**: Ayuda a localizar configuraciones críticas donde el robot pierde grados de libertad o requiere velocidades articulares infinitas para moverse.
    3.  **Análisis de Estática**: A través de su transpuesta ($J^T$), permite calcular los torques necesarios en los motores para soportar una fuerza externa aplicada en la punta del robot.

Este modelo es esencial para asegurar que el movimiento del robot sea fluido, controlado y mecánicamente viable durante su operación.

### Modelado cinemático directo de las velocidades

**Cinemática Directa:** Permite determinar con precisión milimétrica dónde se encuentra la herramienta del robot conociendo únicamente **los ángulos de sus motores**. Es la base para la **visualización en simuladores** y el monitoreo en tiempo real.

Partiendo del modelo de la postura


$$^O \xi_P =^O \xi_P (q)$$

Derivada del modelo


$$\frac{d}{dt}^O \xi_P =\frac{d}{dt}^O \xi_P (q)$$

 Nota: El operador $\frac{d}{\textrm{dt}}$  de la ecuacion es en realidad una instrucción para aplicar la **regla de la cadena**, lo que expande el modelo en derivadas parciales respecto a cada una de las coordenadas articulares ( $\theta$ ).



In [11]:
syms x_dot_O_P y_dot_O_P theta_dot_O_P
syms theta_dot_O_1 theta_dot_1_2 theta_dot_2_3
xi_dot_O_P = [x_dot_O_P; y_dot_O_P; theta_dot_O_P]


$$\frac{d}{dt}^0 \xi_P (q)=\frac{\partial ^0 \xi_P (q)}{\partial ^0 \theta_1 }^0 {\dot{\theta} }_1 +\frac{\partial ^0 \xi_P (q)}{\partial ^1 \theta_2 }^1 {\dot{\theta} }_2 +\frac{\partial ^0 \xi_P (q)}{\partial ^2 \theta_3 }^2 {\dot{\theta} }_3 =\left(\begin{array}{ccc} \frac{\partial ^0 \xi_P (q)}{\partial ^0 \theta_1 } & \frac{\partial ^0 \xi_P (q)}{\partial ^1 \theta_2 } & \frac{\partial ^0 \xi_P (q)}{\partial ^2 \theta_3 } \end{array}\right)\left(\begin{array}{c} ^0 {\dot{\theta} }_1 \newline ^1 {\dot{\theta} }_2 \newline ^2 {\dot{\theta} }_3  \end{array}\right)$$
#### Modelo cinemático directo de las velocidades

La utilidad primordial de este modelo radica en su capacidad para actuar como un "puente" matemático entre el movimiento de los motores y el desplazamiento real de la herramienta en el espacio de trabajo. Sus aplicaciones específicas incluyen:

*   **Control de Trayectorias de Precisión:** Permite programar movimientos donde la velocidad del efector final debe ser constante o seguir un perfil específico, algo vital en tareas industriales como la soldadura por arco o el corte por láser.
*   **Identificación de Configuraciones Críticas:** Facilita la detección de **singularidades cinemáticas**, que son puntos en el espacio donde el robot pierde un grado de libertad o donde los motores requerirían velocidades infinitas para realizar un movimiento mínimo.
*   **Simulación Dinámica:** Es un componente esencial para desarrollar simuladores de alta fidelidad, permitiendo predecir cómo se comportará el robot físicamente antes de ejecutar cualquier comando en el hardware real.


$$^0 {\dot{\mathbf{\xi }} }_P (q)={\mathbf{J}}_{\theta } (q)\dot{q}$$

**Jacobiano**


In [12]:
syms J_theta J_theta_inverse w w_inverse

J_theta

In [13]:
J_theta = simplify((jacobian(xi_O_P_q,[theta_O_1,theta_1_2,theta_2_3])))

In [14]:
q_dot = [theta_dot_O_1; theta_dot_1_2; theta_dot_2_3]

### $$^0 {\dot{\mathbf{\xi }} }_P (q)=\left(\begin{array}{c} {\dot{x} }_{O,P} \newline {\dot{y} }_{O,P} \newline {\dot{\theta} }_{O,P}  \end{array}\right)=\left(\begin{array}{ccc} -L_2 \sin (\theta_{1,2} +\theta_{0,1} )-L_1 \sin (\theta_{0,1} )-L_3 \sin (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} ) & -L_2 \sin (\theta_{1,2} +\theta_{0,1} )-L_3 \sin (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} ) & -L_3 \sin (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} )\newline L_2 \cos (\theta_{1,2} +\theta_{0,1} )+L_1 \cos (\theta_{0,1} )+L_3 \cos (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} ) & L_2 \cos (\theta_{1,2} +\theta_{0,1} )+L_3 \cos (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} ) & L_3 \cos (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} )\newline 1 & 1 & 1 \end{array}\right)\left(\begin{array}{c} {\dot{\theta} }_{0,1} \newline {\dot{\theta} }_{1,2} \newline {\dot{\theta} }_{2,3}  \end{array}\right)$$
#### Vector de velocidades de los eslabones del robot

$${\vec{v} }_P =J_{col1} \cdot {\dot{\theta} }_{0,1} +J_{col2} \cdot {\dot{\theta} }_{1,2} +J_{col3} \cdot {\dot{\theta} }_{2,3}$$

Considerando un analisis en un instante de tiempo determinado o para una configuración dada ( $\theta$ ) (la cadena cinematica en un momento esta "estatica")


 **Eslabón 1:**


In [15]:
syms link_1
link_1= [J_theta(1,1);J_theta(2,1);1]


Las filas 1 y 2 de la primera columna de la matriz Jacobiana ( ${\mathbf{J}}_{v1}$ ) corresponde a la proyeccion del vector velocidad lineal tangente que la articulacion 1 induce sobre el punto P,


Fisicamente, esto equivale a la suma vectorial de las velocidades proyectadas de cada uno de los eslabones sobre dicho punto.


La fila 3 corresponde a la proyeccion del vector de la velocidad angular resultante en el punto debido al movimiento de dicha articulación.".


$${\vec{v} }_{P,1} =\left\lbrack \begin{array}{c} -L_2 \sin (\theta_{1,2} +\theta_{0,1} )-L_1 \sin (\theta_{0,1} )-\sigma_1 \newline L_2 \cos (\theta_{1,2} +\theta_{0,1} )+L_1 \cos (\theta_{0,1} )+\sigma_2 \newline 1 \end{array}\right\rbrack {\dot{\theta} }_{0,1}$$



![Veslabon1](imagenes_ep1/Veslabon1.png)

Fig. 4. ${\mathbf{J}}_{v1}$ ,proyeccion del vector velocidad lineal tangente que la articulacion 1 induce sobre el punto P.

**Eslabón 2 :**


In [16]:
syms link_2
link_2= [J_theta(1,2);J_theta(2,2);1]


Las filas 1 y 2 de la segunda columna de la matriz Jacobiana ( ${\mathbf{J}}_{v2}$ ) corresponde a la proyeccion del vector velocidad lineal tangente que la articulacion 2 induce sobre el punto P.


Fisicamente, esto equivale a la suma vectorial de las velocidades proyectadas de los eslabones que se encuentran a partir de la segunda articulación hacia el extremo final.


La fila 3 corresponde a la contribución de la articulación 2 a la velocidad angular resultante en el punto P.


$${\vec{v} }_{P,2} =\left\lbrack \begin{array}{c} -L_2 \sin (\theta_{1,2} +\theta_{0,1} )-\sigma_1 \newline L_2 \cos (\theta_{1,2} +\theta_{0,1} )+\sigma_2 \newline 1 \end{array}\right\rbrack {\dot{\theta} }_{1,2}$$


![Veslabon2](imagenes_ep1/Veslabon2.png)
  
Fig. 5. ${\mathbf{J}}_{v2}$ ,proyeccion del vector velocidad lineal tangente que la articulacion 2 induce sobre el punto P.

**Eslabón 3:**


In [17]:
syms link_3
link_2= [J_theta(1,3);J_theta(2,3);1]


Las filas 1 y 2 de la tercer columna de la matriz Jacobiana ( ${\mathbf{J}}_{v3}$ ) corresponde a la proyeccion del vector velocidad lineal tangente que la articulacion 3 induce sobre el punto P.


Fisicamente, esto equivale a la suma vectorial de las velocidades proyectadas de los eslabones que se encuentran a partir de la tercera articulación hacia el extremo final.


La fila 3 corresponde a la contribución de la articulación 3 a la velocidad angular resultante en el punto P.


$${\vec{v} }_{P,3} =\left\lbrack \begin{array}{c} -L_3 \sin (\theta_{0,1} +\theta_{1,2} +\theta_{2,3} )\newline L_3 \cos (\theta_{0,1} +\theta_{1,2} +\theta_{2,3} )\newline 1 \end{array}\right\rbrack {\dot{\theta} }_{2,3}$$


![Veslabon3](imagenes_ep1/Veslabon3.png)

  Fig. 6. ${\mathbf{J}}_{v3}$ ,proyeccion del vector velocidad lineal tangente que la articulacion 3 induce sobre el punto P.

**Cálculo de la manipulabilidad**

El **Cálculo de la Manipulabilidad** es una métrica escalar que mide la capacidad de un manipulador robótico para moverse en todas las direcciones de manera uniforme y eficiente desde una configuración articular dada.

    Este valor cuantifica el "volumen" del elipsoide de manipulación en el espacio de velocidades cartesianas.
*   **Utilidad**: 
    1.  **Detección de Singularidades**: Cuando $w$ tiende a cero, el robot se aproxima a una configuración singular donde pierde la capacidad de moverse en una o más direcciones.
    2.  **Optimización del Espacio de Trabajo**: Permite identificar las zonas donde el robot tiene mayor destreza y libertad de movimiento, siendo ideal para ubicar las tareas que requieren mayor precisión.
    3.  **Diseño y Planificación**: Sirve como criterio para comparar diferentes diseños de robots o para elegir la mejor trayectoria posible que mantenga al manipulador alejado de sus límites mecánicos.

Este índice es una medida de la "calidad" de la postura actual del robot respecto a su capacidad de respuesta dinámica.



$$w=|J_{\theta } (q)|$$

In [18]:
w = simplify(det(J_theta))

**Posturas singulares**


$$\begin{array}{l} w=|J_{\theta } (q)|=0\newline ^0 \theta_1 =0,n\pi  \end{array}$$

![singular1](imagenes_ep1/singular1.png)

  Fig. 7. Caso singular 1, ${\theta }$ = 0.


![singular2](imagenes_ep1/singular2.png)

  Fig. 8. Caso singular 2, ${\theta }$ = n${\pi }$ .





### Modelo cinemático inverso de las velocidades

El **Modelo Cinemático Inverso de las Velocidades** es la formulación matemática que permite determinar las velocidades articulares ($\dot{q}_1, \dot{q}_2, \dot{q}_3$) necesarias para que el efector final del robot se desplace con una velocidad lineal y angular específica en el espacio cartesiano.

**Utilidad**

1. **Control de Actuadores:** Es el algoritmo fundamental que indica a cada motor a qué rapidez debe girar para que el robot siga una trayectoria rectilínea o curva con precisión.
2. **Seguimiento de Trayectorias en Tiempo Real:** Permite realizar ajustes dinámicos durante la operación del robot para corregir desviaciones de velocidad de forma instantánea.
3. **Prevención de Sobrecargas:** Ayuda a verificar si la velocidad cartesiana deseada es físicamente posible de alcanzar sin exceder los límites de operación de los motores (límites de RPM).
    
Este modelo constituye el "cerebro" del sistema de control de movimiento, transformando una intención de desplazamiento global en instrucciones individuales para cada articulación.


El modelo cinemetico inverso de las velocidades se obtiene al despejar el vector de velocidades generalizado $\dot{q}$


$$\dot{\mathbf{q}} ={\mathbf{J}}_{\theta }^{-1} (q)^0 {\dot{\mathbf{\xi }} }_P (q)$$


In [19]:
xi_dot_O_P


La matriz $J_{\theta }^{-1} (q)$ representa a la matriz pseudoinversa Jacobiana, para este caso:


In [20]:
J_theta_inverse= simplify(inv(J_theta))


$$\left(\begin{array}{c} {\dot{\theta} }_{0,1} \newline {\dot{\theta} }_{1,2} \newline {\dot{\theta} }_{2,3}  \end{array}\right)=\left(\begin{array}{ccc} \frac{\cos (\theta_{1,2} +\theta_{0,1} )}{L_1 \sin (\theta_{1,2} )} & \frac{\sin (\theta_{1,2} +\theta_{0,1} )}{L_1 \sin (\theta_{1,2} )} & \frac{L_3 \sin (\theta_{2,3} )}{L_1 \sin (\theta_{1,2} )}\newline -\frac{L_2 \cos (\theta_{1,2} +\theta_{0,1} )+L_1 \cos (\theta_{0,1} )}{\sigma_l } & -\frac{L_2 \sin (\theta_{1,2} +\theta_{0,1} )+L_1 \sin (\theta_{0,1} )}{\sigma_l } & -\frac{L_3 (L_1 \sin (\theta_{1,2} +\theta_{2,3} )+L_2 \sin (\theta_{2,3} ))}{\sigma_l }\newline \frac{\cos (\theta_{0,1} )}{L_2 \sin (\theta_{1,2} )} & \frac{\sin (\theta_{0,1} )}{L_2 \sin (\theta_{1,2} )} & \frac{L_3 \sin (\theta_{1,2} +\theta_{2,3} )+L_2 \sin (\theta_{1,2} )}{L_2 \sin (\theta_{1,2} )} \end{array}\right)\left(\begin{array}{c} {\dot{x} }_{O,P} \newline {\dot{y} }_{O,P} \newline {\dot{\theta} }_{O,P}  \end{array}\right)$$


$$|J_{\theta }^{-1} (q)|=1/w$$

In [21]:
w_inverse = simplify(det(J_theta_inverse))

## Modelado cinemático de las aceleraciones
El **Modelado Cinemático de las Aceleraciones** constituye el nivel más alto del análisis cinemático, vinculando los cambios de velocidad en el tiempo con la respuesta física del manipulador.

*   **Representación Física**: Representa las **fuerzas de inercia** y los efectos dinámicos complejos que surgen cuando el robot está en movimiento. Físicamente, este modelo captura:
    *   **Aceleraciones Tangenciales**: El cambio en la rapidez con la que el efector final sigue su trayectoria.
    *   **Aceleraciones Centrípetas y de Coriolis**: Las aceleraciones que aparecen debido a la rotación de los eslabones y al movimiento relativo entre ellos, incluso si los motores giran a velocidad constante.
*   **Utilidad Técnica**: 
**Suavizado de Trayectorias**: Permite diseñar perfiles de movimiento (como el "S-curve") que evitan cambios bruscos en la aceleración (*jerk*), protegiendo la integridad mecánica de los reductores y motores.
**Compensación de Perturbaciones**: Facilita el desarrollo de controladores avanzados que pueden predecir y anular las vibraciones estructurales durante operaciones de alta velocidad.



### Modelo cinemático directo de las aceleraciones

El **Modelo Cinemático Directo de las Aceleraciones** es la formulación matemática que establece la relación entre la aceleración de las articulaciones (motores) y la aceleración resultante en el efector final del robot.

Partiendo del modelo de la velocidad


$$^0 {\dot{\mathbf{\xi }} }_P =^0 {\dot{\mathbf{\xi }} }_P (q)$$

Derivando el modelo


$$\frac{d}{dt}^0 {\dot{\mathbf{\xi }} }_P =\frac{d}{dt}^0 {\dot{\mathbf{\xi }} }_P (q)$$

 Nota: El operador $\frac{d}{\textrm{dt}}$  de la ecuacion es en realidad una instrucción para aplicar la **regla de la cadena**, lo que expande el modelo en derivadas parciales respecto a cada una de las coordenadas articulares ( $\theta$ ).


$$^0 {\ddot{\mathbf{\xi }} }_P =\frac{d}{dt}^0 {\dot{\mathbf{\xi }} }_P (q)$$


$$^0 {\ddot{\mathbf{\xi }} }_P =\frac{\partial ^0 {\dot{\xi} }_P }{\partial \theta_{0,1} }{\dot{\theta} }_{0,1} +\frac{\partial ^0 {\dot{\xi} }_P }{\partial \theta_{1,2} }{\dot{\theta} }_{1,2} +\frac{\partial ^0 {\dot{\xi} }_P }{\partial \theta_{2,3} }{\dot{\theta} }_{2,3} +\frac{\partial ^0 {\dot{\xi} }_P }{\partial {\dot{\theta} }_{0,1} }{\ddot{\theta} }_{0,1} +\frac{\partial ^0 {\dot{\xi} }_P }{\partial {\dot{\theta} }_{1,2} }{\ddot{\theta} }_{1,2} +\frac{\partial ^0 {\dot{\xi} }_P }{\partial {\dot{\theta} }_{2,3} }{\ddot{\theta} }_{2,3}$$


$$^0 {\ddot{\xi} }_P (q)=\left(\begin{array}{ccc} \frac{\partial ^0 {\dot{\xi} }_P (q)}{\partial \theta_{0,1} } & \frac{\partial ^0 {\dot{\xi} }_P (q)}{\partial \theta_{1,2} } & \frac{\partial ^0 {\dot{\xi} }_P (q)}{\partial \theta_{2,3} } \end{array}\right)\left(\begin{array}{c} {\dot{\theta} }_{0,1} \newline {\dot{\theta} }_{1,2} \newline {\dot{\theta} }_{2,3}  \end{array}\right)+\left(\begin{array}{ccc} \frac{\partial ^0 \xi_P (q)}{\partial \theta_{0,1} } & \frac{\partial ^0 \xi_P (q)}{\partial \theta_{1,2} } & \frac{\partial ^0 \xi_P (q)}{\partial \theta_{2,3} } \end{array}\right)\left(\begin{array}{c} {\ddot{\theta} }_{0,1} \newline {\ddot{\theta} }_{1,2} \newline {\ddot{\theta} }_{2,3}  \end{array}\right)$$

$$^0 {\ddot{\xi} }_P =\dot{J} (q)\dot{q} +J(q)\ddot{q}$$

 $\dot{J} (q)\dot{q}$  Es la representacion de la Aceleracion Centripeta y de Coriolis.


Representan cómo cambia la velocidad debido al cambio de configuración del robot mientras se mueve.


 $J(q)\ddot{q}$ Es la representacion de la Aceleracion Tangencial


Representa la relación directa entre las aceleraciones de los motores ( $\theta$ ) y la aceleración instantánea del punto .



In [22]:
syms theta_ddot_O_1 theta_ddot_1_2 theta_ddot_2_3
q_ddot = [theta_ddot_O_1; theta_ddot_1_2; theta_ddot_2_3]

In [ ]:
J_dot_theta = simplify(diff(J_theta,theta_O_1)*theta_dot_O_1 + diff(J_theta,theta_1_2)*theta_dot_1_2 + diff(J_theta,theta_2_3)*theta_dot_2_3)

$ $ {\ddot{\xi} }_P =\left(\begin{array}{ccccccc} \begin{array}{c} -{\dot{\theta} }_{1,2} (L_2 \cos (\theta_{1,2} +\theta_{0,1} )+L_3 \cos (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} ))\newline -{\dot{\theta} }_{0,1} (L_2 \cos (\theta_{1,2} +\theta_{0,1} )+L_1 \cos (\theta_{0,1} )+L_3 \cos (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} ))\newline -L_3 {\dot{\theta} }_{2,3} \cos (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} ) \end{array} & \begin{array}{c} -{\dot{\theta} }_{1,2} (L_2 \cos (\theta_{1,2} +\theta_{0,1} )+L_3 \cos (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} ))\newline -{\dot{\theta} }_{0,1} (L_2 \cos (\theta_{1,2} +\theta_{0,1} )+L_3 \cos (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} ))\newline -L_3 {\dot{\theta} }_{2,3} \cos (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} ) \end{array} & -L_3 \cos (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} )({\dot{\theta} }_{1,2} +{\dot{\theta} }_{2,3} +{\dot{\theta} }_{0,1} )\begin{array}{c} -{\dot{\theta} }_{0,1} (L_2 \sin (\theta_{1,2} +\theta_{0,1} )+L_1 \sin (\theta_{0,1} )+L_3 \sin (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} ))\newline -{\dot{\theta} }_{1,2} (L_2 \sin (\theta_{1,2} +\theta_{0,1} )+L_3 \sin (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} ))\newline -L_3 {\dot{\theta} }_{2,3} \sin (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} ) \end{array} & \begin{array}{c} -{\dot{\theta} }_{1,2} (L_2 \sin (\theta_{1,2} +\theta_{0,1} )+L_3 \sin (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} ))\newline -{\dot{\theta} }_{0,1} (L_2 \sin (\theta_{1,2} +\theta_{0,1} )+L_3 \sin (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} ))\newline -L_3 {\dot{\theta} }_{2,3} \sin (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} ) \end{array} & -L_3 \sin (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} )({\dot{\theta} }_{1,2} +{\dot{\theta} }_{2,3} +{\dot{\theta} }_{0,1} )0 & 0 & 0
\end{array}\right)\left(\begin{array}{c} {\dot{\theta} }_{0,1} \newline {\dot{\theta} }_{1,2} \newline {\dot{\theta} }_{2,3}  \end{array}\right)+\left(\begin{array}{ccc} -L_2 \sin (\theta_{1,2} +\theta_{0,1} )-L_1 \sin (\theta_{0,1} )-L_3 \sin (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} ) & -L_2 \sin (\theta_{1,2} +\theta_{0,1} )-L_3 \sin (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} ) & -L_3 \sin (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} )\newline L_2 \cos (\theta_{1,2} +\theta_{0,1} )+L_1 \cos (\theta_{0,1} )+L_3 \cos (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} ) & L_2 \cos (\theta_{1,2} +\theta_{0,1} )+L_3 \cos (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} ) & L_3 \cos (\theta_{1,2} +\theta_{2,3} +\theta_{0,1} )\newline 1 & 1 & 1 \end{array}\right)\left(\begin{array}{c} {\ddot{\theta} }_{0,1} \newline {\ddot{\theta} }_{1,2} \newline {\ddot{\theta} }_{2,3}  \end{array}\right) $ $


In [24]:
xi_ddot_O_P = simplify(J_theta*q_ddot + J_dot_theta*q_dot)

In [25]:
simplify(J_theta*q_ddot )

In [26]:
simplify(J_dot_theta*q_dot)

### Modelo cinemático inverso de las aceleraciones
El **Modelo Cinemático Inverso de las Aceleraciones** es la formulación matemática que permite determinar las aceleraciones articulares ($\ddot{q}$) necesarias para que el efector final del robot alcance una aceleración cartesiana específica ($\ddot{x}$).

*   **Representación Física**: Representa el comando de aceleración requerido en cada motor para contrarrestar los efectos dinámicos del movimiento. Físicamente, este modelo desglosa la aceleración total necesaria para:
    *   **Vencer la Inercia**: Generar el cambio de rapidez deseado en la herramienta del robot.
    *   **Compensar Efectos No Lineales**: Anular las aceleraciones centrípetas y de Coriolis que surgen de forma natural por la rotación combinada de los eslabones, evitando desviaciones en la trayectoria.

Partiendo de la ecuacion directa


$$^0 {\ddot{\xi} }_P =\dot{J} (q)\dot{q} +J(q)\ddot{q}$$

restamos el término de las aceleraciones de Coriolis y centrípetas ( $\dot{J} (q)\dot{q}$ ) a ambos lados de la ecuación


$$\mathbf{J}(q)\ddot{\mathbf{q}} =^0 {\ddot{\xi} }_P -\dot{\mathbf{J}} (q)\dot{\mathbf{q}}$$

multiplicamos toda la expresión por la inversa del Jacobiano ( ${\mathbf{J}}^{-1} (q)$ )


$$\mathbf{\ddot{q} }={\mathbf{J}}^{-1} (q)(\mathbf{^0 {\ddot{\xi} }_P -\mathbf{\dot{J} }(q)\mathbf{\dot{q} })}$$

donde:


 $\mathbf{\ddot{q} }$: Representa al vector de aceleraciones que deben aplicar los motores en las articulaciones


 ${\mathbf{J}}^{-1} (q)$: La matriz inversa del Jacobiano


 $^0 {\ddot{\xi} }_P$: Es la aceleración deseada en el efector final (punto P)


 $\mathbf{\dot{J} }(q)\mathbf{\dot{q} }$ Representa la compensación necesaria por los efectos de la velocidad actual del robot (Coriolis y Centrípeta).


**Vector de aceleraciones**


In [27]:
vector_aceleraciones= (J_theta_inverse)*(xi_ddot_O_P - (J_dot_theta*q_dot))


## Modelo dinámico por ecuaciones de Eüler\-Lagrange

El **Modelo Dinámico por Ecuaciones de Euler-Lagrange** es la formulación analítica que describe la relación de causa y efecto entre las fuerzas (torques) aplicadas y el movimiento resultante del robot.

*   **Representación Física**: Representa el **balance energético** del sistema. Físicamente, este modelo desglosa cómo se distribuye la energía en el manipulador a través de cuatro componentes clave:
    *   **Inercia ($\mathbf{M}$):** La resistencia de los eslabones a cambiar su estado de movimiento.
    *   **Efectos Centrífugos y de Coriolis ($\mathbf{C}$):** Las fuerzas que aparecen debido a la interacción de las velocidades de los distintos eslabones.
    *   **Fuerzas Gravitacionales ($\mathbf{G}$):** El efecto del peso de la estructura en cada articulación.
    *   **Torques de Entrada ($\tau$):** El esfuerzo que deben realizar los motores para vencer los tres puntos anteriores.
*   **Utilidad**: 
    1.  **Control por Par Calculado**: Es la base para diseñar controladores avanzados que permiten al robot seguir trayectorias de alta precisión compensando automáticamente la gravedad y la inercia.
    2.  **Simulación de Alta Fidelidad**: Permite predecir el comportamiento real del robot en un entorno virtual, incluyendo el consumo de energía y los límites de carga.
    3.  **Dimensionamiento de Motores**: Facilita la selección de actuadores adecuados al calcular los torques máximos requeridos en las condiciones de operación más exigentes.

Se emplearan las siguientes ecuaciones para el cálculo de la energía cinética:


$$k_i =\frac{m_i }{2}{\mathbf{v}}_{C_i }^T {\mathbf{v}}_{C_i } +\frac{1}{2}{\mathbf{\omega }}_{C_i }^T {\mathbf{I}}_{C_i } {\mathbf{\omega }}_{C_i }$$


In [28]:
syms x_1_C1 theta_dot_O_1

v_C1_C1 = [0;x_1_C1*theta_dot_O_1;0]

In [29]:

v_O_C1 = [-x_1_C1*sin(theta_O_1)*theta_dot_O_1;x_1_C1*cos(theta_O_1)*theta_dot_O_1;0]

In [30]:
transpose(v_C1_C1)*v_C1_C1

In [31]:
simplify(transpose(v_O_C1)*v_O_C1)


#### Cálculo de la posición de los centros de masa


In [32]:
syms x_1_C1 x_2_C2 x_3_C3

T_1_C1 = Tij(x_1_C1,0,0,0,0,0)

In [33]:
T_O_C1 = T_O_1*T_1_C1

In [34]:

T_2_C2 = Tij(x_2_C2,0,0,0,0,0)

In [35]:
T_O_C2 = T_O_1*T_1_2*T_2_C2

In [36]:

T_3_C3 = Tij(x_3_C3,0,0,0,0,0)

In [37]:
T_O_C3 = T_O_1*T_1_2*T_2_3*T_3_C3

In [38]:

%Vectores de posición
p_O_C1 = [T_O_C1(1,4);T_O_C1(2,4);T_O_C1(3,4)]

In [39]:
p_O_C2 = simplify([T_O_C2(1,4);T_O_C2(2,4);T_O_C2(3,4)])

In [40]:

p_O_C3 = simplify([T_O_C3(1,4);T_O_C3(2,4);T_O_C3(3,4)])

#### Cálculo de las velocidades


In [41]:
syms theta_dot_O_1 theta_dot_1_2 theta_dot_2_3

v_O_C1 = diff(p_O_C1,theta_O_1)*theta_dot_O_1+diff(p_O_C1,theta_1_2)*theta_dot_1_2+diff(p_O_C1,theta_2_3)*theta_dot_2_3

In [42]:
v_O_C2 = diff(p_O_C2,theta_O_1)*theta_dot_O_1+diff(p_O_C2,theta_1_2)*theta_dot_1_2+diff(p_O_C2,theta_2_3)*theta_dot_2_3

In [43]:
v_O_C3 = diff(p_O_C3,theta_O_1)*theta_dot_O_1+diff(p_O_C3,theta_1_2)*theta_dot_1_2+diff(p_O_C3,theta_2_3)*theta_dot_2_3

#### Cálculo de la velocidades angulares


In [44]:
syms omega_1_1 omega_2_2 omega_3_3
%Propagación para el primer cuerpo
omega_1_1

In [45]:
omega_O_O = [0;0;0]

omega_O_O = 3x1
     0
     0
     0

In [46]:
n_1_1 = [0;0;1]

n_1_1 = 3x1
     0
     0
     1

In [47]:
R_O_1 = [T_O_1(1,1),T_O_1(1,2),T_O_1(1,3);T_O_1(2,1),T_O_1(2,2),T_O_1(2,3);T_O_1(3,1),T_O_1(3,2),T_O_1(3,3)]

In [48]:
R_1_O = transpose(R_O_1)

In [49]:

%Ecuación de propagación
omega_1_1 = R_1_O*omega_O_O+n_1_1*theta_dot_O_1

In [50]:

%Propagación para el segundo cuerpo
omega_2_2

In [51]:
n_2_2 = [0;0;1]

n_2_2 = 3x1
     0
     0
     1

In [52]:
R_1_2 = [T_1_2(1,1),T_1_2(1,2),T_1_2(1,3);T_1_2(2,1),T_1_2(2,2),T_1_2(2,3);T_1_2(3,1),T_1_2(3,2),T_1_2(3,3)]

In [53]:
R_2_1 = transpose(R_1_2)

In [54]:

%Ecuación de propagación
omega_2_2 = R_2_1*omega_1_1+n_2_2*theta_dot_1_2

In [55]:

%Propagación para el tercer cuerpo
omega_3_3

In [56]:
n_3_3 = [0;0;1]

n_3_3 = 3x1
     0
     0
     1

In [57]:
R_2_3 = [T_2_3(1,1),T_2_3(1,2),T_2_3(1,3);T_2_3(2,1),T_2_3(2,2),T_2_3(2,3);T_2_3(3,1),T_2_3(3,2),T_2_3(3,3)]

In [58]:
R_3_2 = transpose(R_2_3)

In [59]:

%Ecuación de propagación
omega_3_3 = R_3_2*omega_2_2+n_3_3*theta_dot_2_3

In [60]:
v_O_C3

#### Defición de los elementos de inercia
La **Definición de los Elementos de Inercia** (Matriz de Inercia $M(q)$) es la descripción de la distribución de la masa de cada eslabón del robot y cómo esta se opone a los cambios en su estado de movimiento.

*   **Representación Física**: Representa la **resistencia dinámica** del sistema ante la aceleración. Físicamente, este componente captura:
    *   **Masas y Momentos de Inercia**: Las propiedades intrínsecas de cada eslabón que determinan cuánta fuerza se requiere para moverlo.
    *   **Configuración Geométrica**: La forma en que la inercia percibida por los motores cambia dependiendo de la postura actual del robot (por ejemplo, es más difícil mover el robot cuando sus brazos están totalmente extendidos que cuando están contraídos).
*   **Utilidad**: 
    1.  **Cálculo de Fuerzas de Movimiento**: Es esencial para determinar los torques necesarios que deben entregar los motores para acelerar los eslabones desde el reposo.
    2.  **Acoplamiento Dinámico**: Permite modelar cómo el movimiento de una articulación afecta a las demás debido a la transferencia de energía a través de la estructura física.
    3.  **Simulación de Impactos y Cargas**: Facilita el análisis de cómo el robot reacciona ante cargas externas o cambios bruscos de dirección, asegurando que los actuadores no se saturen.

En términos matemáticos, esta matriz debe ser siempre **simétrica y definida positiva**, garantizando que el sistema tenga una energía cinética coherente con las leyes de la física.

In [61]:
syms g I_xx1 I_yy1 I_zz1 I_xx2 I_yy2 I_zz2 I_xx3 I_yy3 I_zz3
%vector de gravedad

g_v = [0;-g;0]

In [62]:

I_C1 = [I_xx1,0,0;0,I_yy1,0;0,0,I_zz1]

In [63]:
I_C2 = [I_xx2,0,0;0,I_yy2,0;0,0,I_zz2]

In [64]:
I_C3 = [I_xx3,0,0;0,I_yy3,0;0,0,I_zz3]

#### Cáculo de la energía cinética y energía potencial
El cálculo de las energías cinética y potencial constituye el paso fundamental para derivar el Lagrangiano del sistema, permitiendo obtener las ecuaciones de movimiento que rigen el comportamiento dinámico del robot.

*   **Representación Física**: Representa el **estado energético total** del manipulador en cada instante de su movimiento.
    *   **Energía Cinética ($K$):** Es la energía debida al movimiento. Depende de las masas de los eslabones y de sus velocidades lineales y angulares. Físicamente, captura el esfuerzo necesario para mantener el robot en movimiento.
    *   **Energía Potencial ($V$):** Es la energía almacenada debido a la posición de los eslabones en un campo gravitatorio. Físicamente, representa el trabajo necesario para elevar los eslabones desde un nivel de referencia.
*   **Utilidad**: 
    1.  **Construcción del Lagrangiano ($L = K - V$):** Es la base para aplicar el formalismo de Euler-Lagrange y obtener los torques de los motores.
    2.  **Análisis de Estabilidad:** Permite estudiar puntos de equilibrio y el comportamiento del robot bajo la influencia de la gravedad.
    3.  **Eficiencia Energética:** Facilita el cálculo del consumo de potencia del robot para optimizar trayectorias que minimicen el gasto de batería o electricidad.

Matemáticamente, para un robot de $n$ grados de libertad, la energía cinética total es la suma de las energías de cada eslabón $K = \sum K_i$, mientras que la potencial depende de la altura de los centros de masa $V = \sum m_i g h_i$.

In [65]:
syms m_1 m_2 m_3
%energía cinética de cada uno de los cuerpos

k_1 = simplify((m_1/2)*transpose(v_O_C1)*v_O_C1+(1/2)*transpose(omega_1_1)*I_C1*omega_1_1)

In [66]:

k_2 = simplify((m_2/2)*transpose(v_O_C2)*v_O_C2+(1/2)*transpose(omega_2_2)*I_C2*omega_2_2)

In [67]:

k_3 = simplify((m_3/2)*transpose(v_O_C3)*v_O_C3+(1/2)*transpose(omega_3_3)*I_C3*omega_3_3)

In [68]:
% Cáclulo de la energía potencial de cada cuerpo

u_1 = -m_1*transpose(p_O_C1)*g_v

In [69]:
u_2 = -m_2*transpose(p_O_C2)*g_v

In [70]:
u_3 = -m_3*transpose(p_O_C3)*g_v

#### Cálculo del Lagrangeano

El **Cálculo del Lagrangeano** es el proceso de definir la función escalar que sintetiza la dinámica de un sistema mecánico mediante el balance de sus energías.

*   **Representación Física**: Representa la **función de estado energético** del robot. Se define como la diferencia entre la energía cinética total ($K$) y la energía potencial total ($V$) del sistema.

    Físicamente, describe cómo se distribuye la energía en el manipulador en función de sus posiciones ($q$) y velocidades articulares ($\dot{q}$).
*   **Utilidad**: 
    1.  **Obtención de Ecuaciones de Movimiento**: Es el núcleo del formalismo de Euler-Lagrange. Al aplicar las derivadas parciales sobre esta función, se generan automáticamente los torques necesarios para cada articulación.
    2.  **Unificación del Modelo**: Permite tratar sistemas complejos de múltiples eslabones como un solo ente matemático, evitando la necesidad de analizar diagramas de cuerpo libre individuales por cada pieza.
    3.  **Análisis de Sistemas Multicuerpo**: Facilita la inclusión de efectos como la gravedad y la inercia rotacional en una sola expresión analítica coherente.

Este paso es el puente definitivo entre la descripción energética del robot y su comportamiento dinámico real.

In [71]:

La = (k_1+k_2+k_3)-(u_1+u_2+u_3)

#### Cálculo de los pares


In [72]:
syms theta_ddot_O_1 theta_ddot_1_2 theta_ddot_2_3

D_theta1 = diff(La,theta_dot_O_1)

In [73]:

% Cálculo de relación

tao_1 = diff(D_theta1,theta_O_1)*theta_dot_O_1 + diff(D_theta1,theta_1_2)*theta_dot_1_2 + diff(D_theta1,theta_2_3)*theta_dot_2_3 + diff(D_theta1,theta_dot_O_1)*theta_ddot_O_1+ diff(D_theta1,theta_dot_1_2)*theta_ddot_1_2+ diff(D_theta1,theta_dot_2_3)*theta_ddot_2_3-diff(La,theta_O_1)

In [74]:

D_theta2 = diff(La,theta_dot_1_2)

In [75]:
tao_2 = diff(D_theta2,theta_O_1)*theta_dot_O_1 + diff(D_theta2,theta_1_2)*theta_dot_1_2 + diff(D_theta2,theta_2_3)*theta_dot_2_3 + diff(D_theta2,theta_dot_O_1)*theta_ddot_O_1+ diff(D_theta2,theta_dot_1_2)*theta_ddot_1_2+ diff(D_theta2,theta_dot_2_3)*theta_ddot_2_3-diff(La,theta_1_2)

In [76]:

D_theta3 = diff(La,theta_dot_2_3)

In [77]:
tao_3 = diff(D_theta3,theta_O_1)*theta_dot_O_1 + diff(D_theta3,theta_1_2)*theta_dot_1_2 + diff(D_theta3,theta_2_3)*theta_dot_2_3 + diff(D_theta3,theta_dot_O_1)*theta_ddot_O_1+ diff(D_theta3,theta_dot_1_2)*theta_ddot_1_2+ diff(D_theta3,theta_dot_2_3)*theta_ddot_2_3-diff(La,theta_2_3)

#### Cálculo del vector de pares


In [78]:
tao = [tao_1;tao_2;tao_3]

#### Cálculo de la Matriz de inercia

In [79]:
% Cálculo de la matriz de inercia

M1 = subs(tao,[theta_ddot_O_1,theta_ddot_1_2,theta_ddot_2_3,theta_dot_O_1,theta_dot_1_2,theta_dot_2_3,g],[1,0,0,0,0,0,0])

In [80]:
M2 = subs(tao,[theta_ddot_O_1,theta_ddot_1_2,theta_ddot_2_3,theta_dot_O_1,theta_dot_1_2,theta_dot_2_3,g],[0,1,0,0,0,0,0])

In [81]:
M3 = subs(tao,[theta_ddot_O_1,theta_ddot_1_2,theta_ddot_2_3,theta_dot_O_1,theta_dot_1_2,theta_dot_2_3,g],[0,0,1,0,0,0,0])

In [82]:
M_theta = collect([M1 M2 M3],[m_1,m_2,m_3])


#### Cálculo del vector de pares por efectos de Coriolis

In [83]:
V_theta = subs(tao,[theta_ddot_O_1,theta_ddot_1_2,theta_ddot_2_3,theta_dot_O_1,theta_dot_1_2,theta_dot_2_3,g],[0,0,0,theta_dot_O_1,theta_dot_1_2,theta_dot_2_3,0])


#### Cálculo del vector de pares por efectos de la gravedad

In [84]:
G_theta = subs(tao,[theta_ddot_O_1,theta_ddot_1_2,theta_ddot_2_3,theta_dot_O_1,theta_dot_1_2,theta_dot_2_3,g],[0,0,0,0,0,0,g])

### Cálculo del modelo  Dinámico directo

Este modelo se usa principalmente para la simulacion del movimiento del robot cuando se aplican torques conocidos.


Se usa para simulacion, responde a la pregunta "si aplico estos torques, ¿como se va a mover el robot?"


$$\ddot{\mathbf{q}} =\mathbf{M}(q)^{-1} \left\lbrack \tau_{\theta } -\mathbf{V}(q,\dot{q} )-\mathbf{G}(q)\right\rbrack$$

donde:


 $\ddot{\mathbf{q}} \triangleq$ Vector de aceleraciones de las juntas $\left(\begin{array}{c} {\ddot{\theta} }_{0,1} \newline {\ddot{\theta} }_{1,2} \newline {\ddot{\theta} }_{2,3}  \end{array}\right)$


 $\mathbf{M}(q)^{-1} \triangleq$ Matriz de inercia inversa, Al calcular la inversa, estamos transformando la ecuación de fuerzas en una ecuación de movimiento.


Representa la relación entre las fuerzas (torques) aplicadas y la aceleración resultante.


 $\tau_{\theta } \triangleq$ Vector de pares, contiene los 3 torques de las articulaciones $\left(\begin{array}{c} \tau_{\theta 1} \newline \tau_{\theta 2} \newline \tau_{\theta 3}  \end{array}\right)$


 $\mathbf{V}(\mathbf{q},\mathbf{\dot{q} })\triangleq$ Vector de pares por efectos de Coriolis


 $\mathbf{G}(\mathbf{q})\triangleq$ Vector de pares por efectos de la gravedad


In [85]:
M_theta_inverse= simplify(inv(M_theta))

### Cálculo del modelo  Dinámico inverso

Este modelo se usa principalmente para calcular los pares (torques) necesarios para lograr un movimiento deseado.


$$\tau_{\theta } =\mathbf{M}(q)\ddot{\mathbf{q}} +\mathbf{V}(q,\dot{q} )+\mathbf{G}(q)$$

 $\tau_{\theta } \triangleq$ Vector de pares, contiene los 3 torques de las articulaciones $\left(\begin{array}{c} \tau_{\theta 1} \newline \tau_{\theta 2} \newline \tau_{\theta 3}  \end{array}\right)$


 $\mathbf{M}(q)\triangleq$ Matriz de inercia. Al ser un robot de 3 eslabones,representa la "resistencia" que ofrece el robot a ser acelerado.


la matriz $\mathbf{M}(q)$ es una matriz de  3x3 cuyos términos dependen de los ángulos de las articulaciones $\left(\begin{array}{c} ~\theta_{0,1} \newline \theta_{1,2} \newline \theta_{2,3}  \end{array}\right)$


 $\ddot{\mathbf{q}} \triangleq$ Vector de aceleraciones de las juntas $\left(\begin{array}{c} {\ddot{\theta} }_{0,1} \newline {\ddot{\theta} }_{1,2} \newline {\ddot{\theta} }_{2,3}  \end{array}\right)$


 $\mathbf{V}(\mathbf{q},\mathbf{\dot{q} })\triangleq$ Vector de pares por efectos de Coriolis


 $\mathbf{G}(\mathbf{q})\triangleq$ Vector de pares por efectos de la gravedad



### Conclusión

El análisis matemático desarrollado en este reporte establece una base teórica robusta para la comprensión y operación de un manipulador robótico de configuración **RRR**, que facilita la comprensión física de su funcionamiento en un entorno real. Permitiendo extraer las siguientes conclusiones clave:

*   **Determinación de la Postura:** (Localización Espacial) La aplicación sistemática de la **convención de Denavit-Hartenberg** permitió derivar las ecuaciones de cinemática directa de manera analítica. Este procedimiento garantiza que la **posición** y la **orientación** del efector final queden perfectamente definidas en función de los ángulos de las articulaciones, resolviendo la complejidad de un sistema con tres grados de libertad.
*   **Análisis Cinemático** (Movimiento y Fuerza)  El cálculo de la **matriz Jacobiana** es fundamental para relacionar las velocidades articulares con las del espacio cartesiano. Matemáticamente, esto permite identificar las **singularidades** (regiones donde el robot pierde movilidad) y establecer los límites operativos necesarios para mantener el control total del brazo. En términos físicos, identifica la rapidez de desplazamiento y las **singularidades**  donde el brazo podría bloquearse o perder fuerza, funcionando como un mapa de agilidad y limitaciones de movimiento.
*   **Modelo Dinámico):** (Esfuerzos y Cargas) Mediante el **formalismo de Euler-Lagrange**, se obtuvo el modelo dinámico que describe las fuerzas y torques requeridos para el movimiento. Este modelo cuantifica el esfuerzo de los motores para vencer la gravedad y la inercia, permitiendo definir la fuerza necesaria para manipular cargas sin comprometer la integridad de los actuadores.Este desarrollo es el paso previo indispensable para el diseño de leyes de control, permitiendo predecir el comportamiento del sistema ante diferentes cargas sin necesidad de experimentación física.

Mientras que los **modelos directos** funcionan como el sistema sensorial para **monitorear el estado** real del robot, los **modelos inversos** actúan como el sistema ejecutivo, siendo indispensables para **guiar y controlar** sus acciones de manera inteligente.


*   **Viabilidad del Modelo:** El rigor matemático empleado confirma que el modelo propuesto es coherente y completo. Estos resultados constituyen el modelo matematico teórico necesario para cualquier etapa posterior de simulación numérica o implementación en hardware, cumpliendo con los objetivos primordiales de este estudio.



### Bibliografia

[1] J. J. Craig, *Introduction to Robotics: Mechanics and Control*, 3rd ed. Upper Saddle River, NJ, USA: Pearson, 2017.


[2] E. Peña Medina, "Diapositivas del curso de Robotica" Facultad de Ingeniería, UNAM, Ciudad de México, México, 2024.
